# 基于能量的模型（EBM）理论

EBM 给出的密度是：
\begin{eqnarray*}
p_{\theta}(x) = \frac{\exp(-E_\theta(x))}{Z_\theta},
\end{eqnarray*}
其中 $E_\theta:\mathbb{R}^d \to \mathbb{R}$，$Z_\theta=\int \exp(-E_\theta(x)) dx$。

给定 $\mathbb{R}^d$ 中的样本 $x_1,\dots, x_N$，我们希望找到最大化对数似然的参数 $\theta$：$\max_\theta \sum_{i=1}^N \log p_{\theta}(x_i)$。由于 $Z_\theta$ 是 $\theta$ 的函数，$\log p_{\theta}(x)$ 关于 $\theta$ 的求值和求导涉及一个通常难以处理的积分。

## 用 MCMC 做最大似然训练

我们可以用 MCMC 方法来估计对数似然的梯度：
\begin{eqnarray*}
\nabla_\theta \log p_\theta(x) = -\nabla_\theta E_\theta(x)-\nabla_\theta 
\log Z_\theta.
\end{eqnarray*}
第一项很容易计算（用自动微分）。

### 问题 1（数学）
证明对于第二项，我们有：
\begin{eqnarray*}
\nabla_\theta \log Z_\theta = \mathbb{E}_{p_{\theta}(x)}\left[-\nabla_\theta E_\theta(x)\right] \left(= \int p_{\theta}(x) \left[-\nabla_\theta E_\theta(x)\right] dx \right).
\end{eqnarray*}


这样，我们就可以通过下面的式子得到对数似然梯度的无偏单样本蒙特卡洛估计：
\begin{eqnarray*}
\nabla_\theta \log Z_\theta \approx -\nabla_\theta E_\theta(\tilde{x}),
\end{eqnarray*}
其中 $\tilde{x}\sim p_\theta(x)$，也就是从 EBM 给出的分布中随机采一个样本。因此，我们需要从模型里抽取随机样本。正如课程里解释的，这可以用 Langevin MCMC 完成。首先注意，log 概率关于 $x$ 的梯度（也就是 score）很容易计算：
\begin{eqnarray*}
\nabla_x \log p_\theta(x) = -\nabla_x E_\theta(x) \text{，因为 }  \nabla_x \log Z_\theta = 0.
\end{eqnarray*}
因此，在这种情况下，Langevin MCMC 由下式给出：
\begin{eqnarray*}
x_t = x_{t-1} - \epsilon \nabla_x E_\theta(x_{t-1}) +\sqrt{2\epsilon}z_t, 
\end{eqnarray*}
其中 $z_t\sim \mathcal{N}(0,I)$。当 $\epsilon\to 0$ 且 $t\to \infty$ 时，$x_t$ 将服从 $p_\theta(x)$ 分布（在某些正则条件下）。

在这份作业里，我们将考虑另一种学习流程。

## 分数匹配（Score Matching）

分数（就是上面 Langevin MCMC 里用到的）定义为 $$ s_\theta(x) = \nabla_x\log p_\theta(x) = -\nabla_x E_\theta(x) = -\left( \frac{\partial E_\theta(x)}{\partial x_1},\dots, \frac{\partial E_\theta(x)}{\partial x_d}\right).$$

如果 $p(x)$ 表示（未知的）数据分布，基本的分数匹配目标是最小化：
$$
\mathbb{E}_{p(x)} \|\nabla_x \log p(x) - s_\theta(x)\|^2.
$$

### 问题 2（数学）

这个目标的问题在于，我们无法计算 $\nabla_x \log p(x)$，因为 $p(x)$ 是未知的。我们只能用经验平均来计算关于 $p(x)$ 的（近似）平均。
证明我们可以解决这个问题，因为我们有：
$$
\mathbb{E}_{p(x)} \|\nabla_x \log p(x) - s_\theta(x)\|^2 = c + \mathbb{E}_{p(x)}\left[ \sum_{i=1}^d\left ( \frac{\partial E_\theta(x)}{\partial x_i}\right)^2+2\frac{\partial^2 E_\theta(x)}{\partial x^2_i}\right],
$$
其中 $c$ 是一个常数（不依赖于 $\theta$）。


## 去噪分数匹配（Denoising Score Matching）

分数匹配方法有几个缺点：计算 Hessian 的迹很昂贵，而且在低密度区域分数估计不准，见 [Generative Modeling by Estimating Gradients of the Data Distribution](https://yang-song.net/blog/2021/score/#naive-score-based-generative-modeling-and-its-pitfalls)

去噪分数匹配是一个优雅且可扩展的解决方案。考虑随机变量 $Y = X+\sigma Z$，其中 $X\sim p(x)$，$Z\sim\mathcal{N}(0,I)$。我们用 $p^\sigma(y)$ 表示 $Y$ 的分布。


### 问题 3（数学）
证明
$$
\nabla_y\log p^\sigma(y) = -\frac{1}{\sigma}\mathbb{E}\left[ Z |Y=y\right] = -\frac{1}{\sigma}\mathbb{E}\left[ Z |X+\sigma Z=y\right].
$$


去噪分数匹配的目标现在是
$$
\mathbb{E}_{p^\sigma(y)}\|\nabla_y \log p^\sigma(y) - s_\theta(y)\|^2,
$$
我们将通过对参数 $\theta$ 做梯度下降来最小化它。

### 问题 4（数学）

证明
$$
\mathbb{E}_{p^\sigma(y)}\|\nabla_y \log p^\sigma(y) - s_\theta(y)\|^2 = \mathbb{E}\left\| \frac{Z}{\sigma}+s_\theta(X+\sigma Z)\right\|^2 -C
$$
其中 $C$ 不依赖于 $\theta$。


因此，在实践中，我们将最小化（随机的）损失：
$$
\ell(\theta; x_1,\dots, x_N) = \frac{1}{N} \sum_{i=1}^N \left\| \frac{z_i}{\sigma}+s_\theta(x_i+\sigma z_i)\right\|^2,
$$
其中 $z_i$ 是独立同分布的高斯。由于数据集太大，我们将运行 SGD 算法，也就是分批处理，并用自动微分得到每个 batch 关于 $\theta$ 的梯度。


# 基于能量模型的代码

你将编写一个 EBM，其中能量函数是一个神经网络，封装成一个 python 类，包含以下方法：
- `energy_fn`：接收一批样本 $x_1,\dots,x_B$ 作为参数，计算对应的能量 $E_\theta(x_1),\dots, E_\theta(x_B)$。
- `score`：接收一批样本 $x_1,\dots,x_B` 作为参数，计算对应的分数 $s_\theta(x_1),\dots, s_\theta(x_B)$。
- `sample_langevin`：接收一批起始点 $x_1,\dots, x_B$ 作为参数，默认步长 `eps=0.1`，默认步数 `n_steps=1000`
- `dsm_loss`：接收一批样本 $x_1,\dots,x_B` 作为参数，噪声默认参数 `sigma=0.1`，计算上面相应的（去噪分数匹配）损失。
- `train_epoch`：接收一个 `dataloader` 作为参数，运行一个 epoch 的 SGD 算法。
- `fit`：训练模型多个 epoch。最后一个方法已经提供给你，借助下面不应修改的代码，你可以可视化你的结果。


In [ ]:
import numpy as np
import torch
from torch import nn
import time
import logging
import matplotlib.pyplot as plt
import functools
import os

In [ ]:
# 借自这个仓库
#    https://github.com/kamenbliznashki/normalizing_flows

def sample_2d(dataset, n_samples):

    z = torch.randn(n_samples, 2)

    if dataset == '8gaussians':
        scale = 4
        sq2 = 1/math.sqrt(2)
        centers = [(1,0), (-1,0), (0,1), (0,-1), (sq2,sq2), (-sq2,sq2), (sq2,-sq2), (-sq2,-sq2)]
        centers = torch.tensor([(scale * x, scale * y) for x,y in centers])
        return sq2 * (0.5 * z + centers[torch.randint(len(centers), size=(n_samples,))])

    elif dataset == '2spirals':
        n = torch.sqrt(torch.rand(n_samples // 2)) * 540 * (2 * math.pi) / 360
        d1x = - torch.cos(n) * n + torch.rand(n_samples // 2) * 0.5
        d1y =   torch.sin(n) * n + torch.rand(n_samples // 2) * 0.5
        x = torch.cat([torch.stack([ d1x,  d1y], dim=1),
                       torch.stack([-d1x, -d1y], dim=1)], dim=0) / 3
        return x + 0.1*z

    elif dataset == 'checkerboard':
        x1 = torch.rand(n_samples) * 4 - 2
        x2_ = torch.rand(n_samples) - torch.randint(0, 2, (n_samples,), dtype=torch.float) * 2
        x2 = x2_ + x1.floor() % 2
        return torch.stack([x1, x2], dim=1) * 2

    elif dataset == 'rings':
        n_samples4 = n_samples3 = n_samples2 = n_samples // 4
        n_samples1 = n_samples - n_samples4 - n_samples3 - n_samples2

        # 为了避免第一个点等于最后一个点，在 np 里设 endpoint=False；这里平移了一位
        linspace4 = torch.linspace(0, 2 * math.pi, n_samples4 + 1)[:-1]
        linspace3 = torch.linspace(0, 2 * math.pi, n_samples3 + 1)[:-1]
        linspace2 = torch.linspace(0, 2 * math.pi, n_samples2 + 1)[:-1]
        linspace1 = torch.linspace(0, 2 * math.pi, n_samples1 + 1)[:-1]

        circ4_x = torch.cos(linspace4)
        circ4_y = torch.sin(linspace4)
        circ3_x = torch.cos(linspace4) * 0.75
        circ3_y = torch.sin(linspace3) * 0.75
        circ2_x = torch.cos(linspace2) * 0.5
        circ2_y = torch.sin(linspace2) * 0.5
        circ1_x = torch.cos(linspace1) * 0.25
        circ1_y = torch.sin(linspace1) * 0.25

        x = torch.stack([torch.cat([circ4_x, circ3_x, circ2_x, circ1_x]),
                         torch.cat([circ4_y, circ3_y, circ2_y, circ1_y])], dim=1) * 3.0

        # 随机采样
        x = x[torch.randint(0, n_samples, size=(n_samples,))]

        # 加噪声
        return x + torch.normal(mean=torch.zeros_like(x), std=0.08*torch.ones_like(x))
    elif dataset == 'gaussian':
        return z + 2*torch.ones(2)
    else:
        raise RuntimeError('Invalid `dataset` to sample from.')

def plot_data(
    ax,
    data,
    range_lim=4,
    bins=1000,
    cmap=plt.cm.viridis
):
    rng = [[-range_lim, range_lim], [-range_lim, range_lim]]
    ax.hist2d(data[:,0], data[:, 1], range=rng, bins=bins, cmap=plt.cm.viridis)

def plot_scores(
    ax,
    mesh,
    scores,
    width=0.002
):
    """Plot score field

    Args:
        ax (): canvas
        mesh (np.ndarray): mesh grid
        scores (np.ndarray): scores
        width (float, optional): vector width. Defaults to 0.002
    """
    ax.quiver(mesh[:, 0], mesh[:, 1], scores[:, 0], scores[:, 1], width=width)

def plot_energy(
    ax,
    energy,
    cmap=plt.cm.viridis,
    flip_y=True
):
    if flip_y:
        energy = energy[::-1] # energy = energy[::-1] # 翻转 y
    ax.imshow(energy, cmap=cmap)
    
def plot_score_field(ax, energy_model):
    mesh, scores = sample_score_field(
        energy_model.score,
        device=energy_model.device
    )
    # 画分数
    ax.grid(False)
    ax.axis('off')
    plot_scores(ax, mesh, scores)
    ax.set_title('Estimated scores', fontsize=16)

def plot_energy_field(ax, energy_model):
    energy = sample_energy_field(
        energy_model.energy_fn,
        device=energy_model.device
    )
    # 画能量
    ax.grid(False)
    ax.axis('off')
    plot_energy(ax, energy)
    ax.set_title('Estimated energy', fontsize=16)

def plot_samples(ax, energy_model, steps, eps):
    samples = []
    for i in range(1000):
        x = torch.rand(1000, 2) * 8 - 4
        x = x.to(device=energy_model.device)
        x = energy_model.sample_langevin(
            x,
            n_steps=steps,
            eps=eps
        ).detach().cpu().numpy()
        samples.append(x)
    samples = np.concatenate(samples, axis=0)
    # 画能量
    ax.grid(False)
    ax.axis('off')
    plot_data(ax, samples)
    ax.set_title('Sampled data', fontsize=16)

def sample_score_field(
    score_fn,
    range_lim=4,
    grid_size=50,
    device='cpu'
):
    """Sampling score field from an energy model

    Args:
        score_fn (callable): a score function with the following sign
            func(x: torch.Tensor) -> torch.Tensor
        range_lim (int, optional): Range of x, y coordimates. Defaults to 4.
        grid_size (int, optional): Grid size. Defaults to 50.
        device (str, optional): torch device. Defaults to 'cpu'.
    """
    mesh = []
    x = np.linspace(-range_lim, range_lim, grid_size)
    y = np.linspace(-range_lim, range_lim, grid_size)
    for i in x:
        for j in y:
            mesh.append(np.asarray([i, j]))
    mesh = np.stack(mesh, axis=0)
    x = torch.from_numpy(mesh).float()
    x = x.to(device=device)
    scores = score_fn(x.detach()).detach()
    scores = scores.cpu().numpy()
    return mesh, scores

def sample_energy_field(
    energy_fn,
    range_lim=4,
    grid_size=1000,
    device='cpu'
):
    """Sampling energy field from an energy model

    Args:
        energy_fn (callable): an energy function with the following sign
            func(x: torch.Tensor) -> torch.Tensor
        range_lim (int, optional): range of x, y coordinates. Defaults to 4.
        grid_size (int, optional): grid size. Defaults to 1000.
        device (str, optional): torch device. Defaults to 'cpu'.
    """
    energy = []
    x = np.linspace(-range_lim, range_lim, grid_size)
    y = np.linspace(-range_lim, range_lim, grid_size)
    for i in y:
        mesh = []
        for j in x:
            mesh.append(np.asarray([j, i]))
        mesh = np.stack(mesh, axis=0)
        inputs = torch.from_numpy(mesh).float()
        inputs = inputs.to(device=device)
        e = energy_fn(inputs.detach()).detach()
        e = e.view(grid_size).cpu().numpy()
        energy.append(e)
    energy = np.stack(energy, axis=0) # energy = np.stack(energy, axis=0) # (grid_size, grid_size)
    return energy
    
    
def visualize(energy_model, data, steps, eps):
    fig, axs = plt.subplots(figsize=(24, 6), ncols=4)
    # 画数据样本
    axs[0].grid(False)
    axs[0].axis('off')
    plot_data(axs[0], data)
    axs[0].set_title('Ground truth data', fontsize=16)
    plot_samples(axs[1], energy_model, steps, eps)
    plot_energy_field(axs[2], energy_model)
    plot_score_field(axs[3], energy_model)
    for ax in axs:
        ax.set_box_aspect(1)
    plt.tight_layout()
    plt.show()

## 数据集

下面是一个直方图，展示用来学习 EDM 的最终数据集。你可以选择合适的数据集名称来更换数据集。


In [ ]:
data_name = 'checkerboard'
size = 1000000
data_np = sample_2d(dataset=data_name, n_samples=size).numpy()
plot_data(plt, data_np)

## 玩具例子：高斯

在写完整的 EBM 之前，我们先处理一个简单的例子，看看怎么自动计算分数，然后运行 SGD。

首先，下面的代码展示如何使用 [`torch.autograd.grad`](https://pytorch.org/docs/stable/generated/torch.autograd.grad.html) 计算分数：


In [ ]:
theta = torch.zeros(2)
data = torch.randn(10,2)

In [ ]:
x = data
x = x.requires_grad_()
logp = -torch.sum((x-theta)**2)/2

In [ ]:
score = torch.autograd.grad(logp, x)[0]

In [ ]:
def my_score(x, theta=theta):
    return theta-x

In [ ]:
torch.allclose(score, my_score(data))

然而，对 SGD 算法，我们想对参数 $\theta$ 再求一次导数。让我们用一个下面定义的假损失看看会发生什么：


In [ ]:
x = data
x = x.requires_grad_()
theta = theta.requires_grad_()
logp = -torch.sum((x-theta)**2)/2
score = torch.autograd.grad(logp, x)[0]
fake_loss = score.norm()**2

In [ ]:
theta.requires_grad

In [ ]:
theta.grad

下面的代码会产生一个错误，因为默认情况下，PyTorch 不会为导数保留计算图。


In [ ]:
# 会产生一个错误
#fake_loss.backward()

使用 `create_graph=True`，我们告诉 PyTorch 我们还会对分数（它已经是关于 $x$ 的导数）再求一次导数。


In [ ]:
x = data
x = x.requires_grad_()
theta = theta.requires_grad_()
logp = -torch.sum((x-theta)**2)/2
score = torch.autograd.grad(logp, x, create_graph=True)[0]
fake_loss = score.norm()**2

In [ ]:
fake_loss.backward()

In [ ]:
theta.grad

In [ ]:
torch.allclose(theta.grad, 2*my_score(x).sum(0))

### 问题 5（代码）

用计算能量、分数，最后用 Langevin MCMC 采样的函数，重新编写上面的例子。所有函数都应该支持 batch。


In [ ]:
theta = torch.zeros(2)
def energy_fn(x, theta=theta):
    # 在这里写你的代码

In [ ]:
# 应该为 True
torch.allclose(torch.sum(energy_fn(x)), -logp)

In [ ]:
def score(x, fn = energy_fn):
    # 在这里写你的代码

In [ ]:
# 应该为 True
torch.allclose(score(x), my_score(x))

In [ ]:
def sample_langevin(x, score=score, eps=0.1, n_steps=1000):
    # 在这里写你的代码

In [ ]:
data_name = 'gaussian'
size = 1000000
data_np = sample_2d(dataset=data_name, n_samples=size).numpy()
dtype=torch.float32
data_t = torch.from_numpy(data_np).type(dtype)

In [ ]:
data_simu = sample_langevin(data_t, eps=0.1, n_steps=100)

In [ ]:
fig, axs = plt.subplots(figsize=(24, 6), ncols=2)
plot_data(axs[0], data_np)
axs[0].set_title('Ground truth data', fontsize=16)
plot_data(axs[1], data_simu.detach().numpy())
axs[1].set_title('Langevin sampling (before fit)', fontsize=16);

这里，你应该会看到真实值和 Langevin 采样之间的不匹配，这是意料之中的，因为你还没有编写训练步骤。真实值的高斯均值是 $(2,2)$，而你的（高斯）EBM 的默认均值是 $(0,0)$。所以现在你将编写模型的训练。

### 问题 6（代码）

现在把你上面的代码重新打包成一个 python 类，包含这些方法：
- `energy_fn`：接收一批样本 $x_1,\dots,x_B$ 作为参数，计算对应的能量 $(x_1-\theta)^2/2,\dots, (x_B-\theta)^2/2$。
- `score`：接收一批样本 $x_1,\dots,x_B$ 作为参数，计算对应的分数 $s_\theta(x_1),\dots, s_\theta(x_B)$。
- `sample_langevin`：接收一批起始点 $x_1,\dots, x_B$ 作为参数，默认步长 `eps=0.1`，默认步数 `n_steps=1000`
- `dsm_loss`：接收一批样本 $x_1,\dots,x_B$ 作为参数，噪声默认参数 `sigma=0.1`，计算相应的（去噪分数匹配）损失。
- `train_epoch`：接收一个 `dataloader` 作为参数，运行一个 epoch 的 SGD 算法。
- `fit`：训练模型多个 epoch（下面已经给你）。


In [ ]:
class Energy_Gaussian():
    def __init__(self, theta=theta, learning_rate =1e-3):
        self.theta = theta.requires_grad_()
        self.learning_rate = learning_rate
        self.optimizer = torch.optim.Adam([self.theta], lr=self.learning_rate)
        
        
    def energy_fn(self, x):
        # 在这里写你的代码
    
    def score(self, x):
        # 在这里写你的代码
    
    def sample_langevin(self, x, eps=0.1, n_steps=1000):
        # 在这里写你的代码
    
    def dsm_loss(self, x, sigma=0.1):
        # 在这里写你的代码
    
    def train_epoch(self, dataloader):
        all_losses = []
        # 在这里写你的代码
        m_loss = np.mean(all_losses).astype(np.float32)
        return m_loss
    
    def fit(self, train_dataloader, 
            n_epochs = 1,
            log_freq = 1,
            vis_freq = 1,
            vis_callback = None):
        
        total_epochs = n_epochs
        num_epochs = 0

        for epoch in range(n_epochs):
            num_epochs += 1
            # 训练一个 epoch
            loss = self.train_epoch(train_dataloader)
            
            if (log_freq is not None) and (num_epochs % log_freq == 0):
                print(
                    f"[Epoch {num_epochs}/{total_epochs}]: loss: {loss}  theta: {self.theta.data}"
                )

            if (vis_callback is not None) and (num_epochs % vis_freq == 0):
                print("Visualizing")
                vis_callback(self)
        pass

In [ ]:
dtype=torch.float32
data = torch.from_numpy(data_np).type(dtype)
loader_train = torch.utils.data.DataLoader(data, batch_size=100, shuffle=True)

In [ ]:
energy_gauss = Energy_Gaussian()

In [ ]:
energy_gauss.fit(loader_train, vis_callback = functools.partial(
            visualize,
            data = data_np,
            steps = 100,
            eps = 0.1
        ))

可以看到，仅仅一个 epoch 之后，我们就有了不错的估计。现在，我们将处理更有挑战性的数据集


## 棋盘格的能量模型

下面给出计算能量所用的神经网络：


In [ ]:
hidden_units = 128
my_mlp = nn.Sequential(
            nn.Linear(2, hidden_units),
            nn.Softplus(),
            nn.Linear(hidden_units, hidden_units),
            nn.Softplus(),
            nn.Linear(hidden_units, 1),
        )

### 问题 7（代码）

现在把上面的例子改编到这个更通用的设定，创建一个 PyTorch 模块。


In [ ]:
# --- 能量模型 ---
class Energy(nn.Module):
    def __init__(self, net, learning_rate =1e-3, device = 'cuda'):
        super().__init__()
        self.device = device
        self.net = net.to(device=self.device)
        self.learning_rate = learning_rate
        self.optimizer = torch.optim.Adam(self.net.parameters(), lr=self.learning_rate)

    def energy_fn(self, x):
        return self.net(x)

    def score(self, x):
        # 在这里写你的代码
    
    def sample_langevin(self, x, eps=0.1, n_steps=1000):
        # 在这里写你的代码
    
    def dsm_loss(self, x, sigma=0.1):
        # 在这里写你的代码
    
    def train_epoch(self, dataloader):
        # 在这里写你的代码
    
    def fit(self, train_dataloader, 
            n_epochs = 5,
            batch_size = 100,
            log_freq = 1,
            vis_freq = 1,
            vis_callback = None):
        
        total_epochs = n_epochs
        num_epochs = 0

        for epoch in range(n_epochs):
            num_epochs += 1
            # 训练一个 epoch
            loss = self.train_epoch(train_dataloader)
            
            if (log_freq is not None) and (num_epochs % log_freq == 0):
                print(
                    f"[Epoch {num_epochs}/{total_epochs}]: loss: {loss}"
                )

            if (vis_callback is not None) and (num_epochs % vis_freq == 0):
                print("Visualizing")
                self.net.eval()
                vis_callback(self)
                self.net.train()
        pass
        

In [ ]:
data_name = 'checkerboard'
size = 1000000
data_np = sample_2d(dataset=data_name, n_samples=size).numpy()

In [ ]:
dtype=torch.float32
data = torch.from_numpy(data_np).type(dtype)
loader_train = torch.utils.data.DataLoader(data, batch_size=100, shuffle=True)

In [ ]:
energy_model = Energy(my_mlp)

In [ ]:
energy_model.fit(loader_train, vis_callback = functools.partial(
            visualize,
            data = data_np,
            steps = 100,
            eps = 0.01
        ))